# 0. Goals 04_Modeling V2

- Transform Categorical Column to Numerical
- Re Modelling Using New Feature

In [1]:
import pandas as pd
import numpy as np
import time
import joblib

from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.feature_selection import mutual_info_classif
from catboost import CatBoostClassifier

In [2]:
path_data_train = r'..\Data\result\03_Exploratory_Data_Analysis_Train.csv'

In [3]:
df_train = pd.read_csv(path_data_train)

# 1. Transform Categorical Column to Numerical

In [4]:
# Cek kolom yang non numerical untuk di encoding

df_train.select_dtypes(include=['str']).info()

<class 'pandas.DataFrame'>
RangeIndex: 255387 entries, 0 to 255386
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   Amount_Bucket      255387 non-null  str  
 1   Delta_Time_Bucket  255387 non-null  str  
 2   Bucket_V14         255387 non-null  str  
 3   Bucket_V10         255387 non-null  str  
 4   Bucket_V12         255387 non-null  str  
 5   Bucket_V11         255387 non-null  str  
 6   Bucket_V3          255387 non-null  str  
 7   Bucket_V9          255387 non-null  str  
 8   Bucket_V7          255387 non-null  str  
 9   Bucket_V1          255387 non-null  str  
 10  Bucket_V26         255387 non-null  str  
 11  Bucket_Amount      255387 non-null  str  
 12  Bucket_V24         255387 non-null  str  
 13  Bucket_V22         255387 non-null  str  
 14  Bucket_V8          255387 non-null  str  
dtypes: str(15)
memory usage: 29.2 MB


In [5]:
# Karena kolom 'bucket delta time' dan kolom 'bucket percent amount' itu memperahtikan urutan (ordinal) maka pakainya ordinal encoder

model_encoder = OrdinalEncoder()
df_train[df_train.select_dtypes(include=['str']).columns.tolist()] = model_encoder.fit_transform(df_train[df_train.select_dtypes(include=['str']).columns.tolist()])

In [6]:
# export model encodernya pakai joblib

joblib.dump(model_encoder, r'..\Output\Model\model_encoder_ordinal.pkl')

['..\\Output\\Model\\model_encoder_ordinal.pkl']

# Split Dataset

In [7]:
X = df_train.drop(columns='Class')
y = df_train['Class']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [9]:
train = X_train.copy()
train['Class'] = y_train

test = X_test.copy()
test['Class'] = y_test

# Re-Train Model

In [10]:
# Dengan AUPRC 0.849
best_ETC_params = {
    'n_estimators'        : 296, 
    'max_depth'           : 37, 
    'min_samples_split'   : 11, 
    'min_samples_leaf'    : 3, 
    'criterion'           : 'entropy', 
    'class_weight'        : None
}

# Dengan AUPRC 0.852
best_RFC_params = {
    'n_estimators'        : 691, 
    'max_depth'           : 23, 
    'min_samples_split'   : 4, 
    'min_samples_leaf'    : 4, 
    'criterion'           : 'log_loss', 
    'class_weight'        : 'balanced_subsample', 
    'bootstrap'           : False
}

# Dengan AUPRC 0.853
best_CB_params = {
    'iterations': 594, 
    'depth': 9, 
    'learning_rate': 0.010639915618971553, 
    'l2_leaf_reg': 1.1199438541669806, 
    'random_strength': 0.8875933831253247
}

In [11]:
etc_model_v2 = ExtraTreesClassifier(
    **best_ETC_params,
    random_state=42,
    n_jobs=-1
)

rfc_model_v2 = RandomForestClassifier(
    **best_RFC_params,
    random_state=42,
    n_jobs=-1
)

cb_model_v2 = CatBoostClassifier(
    **best_CB_params,
    verbose=False,
    thread_count=-1
)

## Extra Trees Classifier

In [12]:
start_etc_train = time.perf_counter()

etc_model_v2.fit(X_train, y_train)

end_etc_train = time.perf_counter()

training_etc_model_time = end_etc_train - start_etc_train

### Test

In [13]:
start_etc_test = time.perf_counter()

etc_pred = etc_model_v2.predict_proba(X_test)[:, 1]

end_etc_test = time.perf_counter()

test_etc_model_time = end_etc_test - start_etc_test

In [14]:
etc_v2_auprc = average_precision_score(y_test, etc_pred)

In [15]:
print(f"Training time Extra Trees Classifier Model \t: {training_etc_model_time:.4f} seconds")
print(f"Test (Inference) time Extra Trees Classifier Model \t: {test_etc_model_time:.4f} seconds")

print()
print('Score AUPRC Extra Trees Classifier Model \t:',  etc_v2_auprc)

Training time Extra Trees Classifier Model 	: 18.5606 seconds
Test (Inference) time Extra Trees Classifier Model 	: 0.2478 seconds

Score AUPRC Extra Trees Classifier Model 	: 0.8763500792155938


In [16]:
importance_etc = pd.DataFrame({
    "feature": X_train.columns,
    "importance": etc_model_v2.feature_importances_
}).sort_values("importance",ascending=False).reset_index(drop=True)

## Random Forest Classifier

In [17]:
start_rfc_train = time.perf_counter()

rfc_model_v2.fit(X_train, y_train)

end_rfc_train = time.perf_counter()

training_rfc_model_time = end_rfc_train - start_rfc_train

### Test

In [18]:
start_rfc_test = time.perf_counter()

rfc_pred = rfc_model_v2.predict_proba(X_test)[:, 1]

end_rfc_test = time.perf_counter()

test_rfc_model_time = end_rfc_test - start_rfc_test

In [19]:
rfc_v2_auprc = average_precision_score(y_test, rfc_pred)

In [20]:
print(f"Training time Random Forest Classifier Model: {training_rfc_model_time:.4f} seconds")
print(f"Test (Inference) time Random Forest Classifier Model: {test_rfc_model_time:.4f} seconds")

print()
print('Score AUPRC Random Forest Classifier Model \t:',  rfc_v2_auprc)

Training time Random Forest Classifier Model: 181.9012 seconds
Test (Inference) time Random Forest Classifier Model: 1.0987 seconds

Score AUPRC Random Forest Classifier Model 	: 0.8930574613611661


In [21]:
importance_rfc = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rfc_model_v2.feature_importances_
}).sort_values("importance",ascending=False).reset_index(drop=True)

## CatBoost Classifier

In [22]:
start_cb_train = time.perf_counter()

cb_model_v2.fit(X_train, y_train)

end_cb_train = time.perf_counter()

training_cb_model_time = end_cb_train - start_cb_train

### Test

In [23]:
start_cb_test = time.perf_counter()

cb_pred = cb_model_v2.predict_proba(X_test)[:, 1]

end_cb_test = time.perf_counter()

test_cb_model_time = end_cb_test - start_cb_test

In [24]:
cb_v2_auprc = average_precision_score(y_test, cb_pred)

In [25]:
print(f"Training time CatBoost Classifier Model: {training_cb_model_time:.4f} seconds")
print(f"Test (Inference) time CatBoost Classifier Model: {test_cb_model_time:.4f} seconds")

print()
print('Score AUPRC CatBoost Classifier Model \t:',  cb_v2_auprc)

Training time CatBoost Classifier Model: 144.2571 seconds
Test (Inference) time CatBoost Classifier Model: 0.0492 seconds

Score AUPRC CatBoost Classifier Model 	: 0.89355503588768


In [26]:
importance_cb = pd.DataFrame({
    "feature": X_train.columns,
    "importance": cb_model_v2.get_feature_importance()
}).sort_values("importance",ascending=False).reset_index(drop=True)

# Feature Importance V2

## Mutual Information

In [27]:
mi = mutual_info_classif(X_train, y_train, random_state=42)

In [28]:
mi_df = pd.DataFrame({
    "features" : X_train.columns,
    "mi" : mi
})

In [29]:
mi_df.sort_values(by="mi", ascending=False)

,features,mi
41,Bucket_Amount,0.060756
32,Bucket_V14,0.057514
37,Bucket_V9,0.057402
38,Bucket_V7,0.052995
34,Bucket_V12,0.045890
39,Bucket_V1,0.041353
33,Bucket_V10,0.036160
42,Bucket_V24,0.034839
36,Bucket_V3,0.034613
30,Amount_Bucket,0.033066


## Extra Trees Classifier Feature Importance

In [30]:
importance_etc

,feature,importance
0,V14,0.153474
1,V17,0.115509
2,V12,0.108862
3,V11,0.078900
4,V16,0.075060
5,V10,0.072600
6,V4,0.054836
7,V3,0.046050
8,V18,0.035698
9,V9,0.025678


## Random Forest Classifier Feature Importance

In [31]:
importance_rfc

,feature,importance
0,V14,1.483114e-01
1,V10,1.099894e-01
2,V17,1.005569e-01
3,V4,9.335463e-02
4,V12,9.262262e-02
5,V11,6.734674e-02
6,V3,4.468713e-02
7,V16,3.679523e-02
8,V7,3.181859e-02
9,V2,2.530287e-02


## Random Forest Classifier Feature Importance

In [32]:
importance_cb

,feature,importance
0,V26,7.389334
1,V4,7.280685
2,V1,6.185765
3,V14,5.245097
4,V12,4.536895
5,V8,4.422247
6,Amount,4.190012
7,V15,3.813039
8,Time,3.690124
9,V20,3.443574


# Export Model V2

In [34]:
cb_model_v2.save_model("../Output/Model/model_v2_catboost.cbm")
joblib.dump(etc_model_v2,"../Output/Model/model_v2_extra_trees.pkl")
joblib.dump(rfc_model_v2,"../Output/Model/model_v2_random_forest.pkl")

['../Output/Model/model_v2_random_forest.pkl']

# Summary V1 vs V2

Berikut merupakan perbandingan Performa dan Detail model V1 vs V2

<table>
    <tr>
        <th rowspan="2">Model</th>
        <th colspan="2">AUPRC</th>
        <th colspan="2">Train Time</th>
        <th colspan="2">Test Time</th>
      </tr>
    <tr>
        <th>V1</th>
        <th>V2</th>
        <th>V1</th>
        <th>V2</th>
        <th>V1</th>
        <th>V2</th>
    </tr>
    <tr>
        <td>Extra Trees</td>
        <td>0.81</td>
        <td>0.87</td>
        <td>30.5s</td>
        <td>18.5s</td>
        <td>0.74s</td>
        <td>0.2s</td>
    </tr>
    <tr>
        <td>Random Forest</td>
        <td>0.82</td>
        <td>0.89</td>
        <td>226.5s</td>
        <td>181.9s</td>
        <td>0.46s</td>
        <td>1.0s</td>
    </tr>
    <tr>
        <td>CatBoost</td>
        <td>0.81</td>
        <td>0.89</td>
        <td>29.8s</td>
        <td>144.2s</td>
        <td>0.05s</td>
        <td>0.04s</td>
    </tr>
</table>